In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import gaussian_kde, chi2
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, cohen_kappa_score
import matplotlib.pyplot as plt

# General plotting style
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

# Output folders
os.makedirs("figures", exist_ok=True)
os.makedirs("tables", exist_ok=True)

print("Working directory:", Path('.').resolve())


Working directory: C:\Users\eldeivid\Documents\DCC\1\Math\statistics


## 1. Data loading and preprocessing

In [ ]:

df = pd.read_csv("dataset.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (2111, 17)


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


### 1.1 Obesity level recoding and variable classification

In [ ]:
# Recode NObeyesdad into four obesity levels
mapping = {
    'Insufficient_Weight': 'Insufficient',
    'Normal_Weight': 'Normal',
    'Overweight_Level_I': 'Overweight',
    'Overweight_Level_II': 'Overweight',
    'Obesity_Type_I': 'Obesity',
    'Obesity_Type_II': 'Obesity',
    'Obesity_Type_III': 'Obesity',
}

df["Obesity4"] = df["NObeyesdad"].map(mapping)
df["Obesity4"] = pd.Categorical(
    df["Obesity4"],
    categories=["Insufficient", "Normal", "Overweight", "Obesity"],
    ordered=True
)


continuous = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
binary = ["FAVC", "SMOKE", "SCC"]
categorical = ["Gender", "family_history_with_overweight", "CAEC", "CALC", "MTRANS"]

all_vars = continuous + binary + categorical

print("Continuous:", continuous)
print("Binary:", binary)
print("Categorical:", categorical)


df["Obesity4"].value_counts().sort_index()


Continuous: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Binary: ['FAVC', 'SMOKE', 'SCC']
Categorical: ['Gender', 'family_history_with_overweight', 'CAEC', 'CALC', 'MTRANS']


Obesity4
Insufficient    272
Normal          287
Overweight      580
Obesity         972
Name: count, dtype: int64

## 2. Helper functions (PMF/PDF, CDF, CIs, ROC, etc.)

In [4]:
def pmf_cdf(s: pd.Series):
    """Empirical PMF and CDF for discrete/categorical variables."""
    v = s.dropna()
    c = v.value_counts().sort_index()
    pmf = c / c.sum()
    return pmf, pmf.cumsum()


def kde_pdf_cdf(s: pd.Series, grid: int = 200):
    """KDE-based PDF and CDF for continuous variables."""
    d = s.dropna().values
    if len(d) < 5:
        x = np.array(d)
        if len(x) == 0:
            return x, np.array([]), np.array([]), None
        pdf = np.ones_like(x) / len(x)
        cdf = np.linspace(0, 1, len(x))
        return x, pdf, cdf, None

    k = gaussian_kde(d)
    mn, mx = d.min(), d.max()
    pad = 0.05 * (mx - mn) if mx > mn else 1.0
    x = np.linspace(mn - pad, mx + pad, grid)
    pdf = k(x)
    # Normalize PDF numerically to integrate to 1
    pdf = pdf / np.trapz(pdf, x)
    cdf = np.cumsum(pdf)
    cdf = cdf / cdf[-1]
    return x, pdf, cdf, k


def mean_ci(x, alpha=0.05):
    """Sample mean and CI using t-distribution."""
    x = np.asarray(x)
    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan
    m = x.mean()
    s = x.std(ddof=1)
    t = stats.t.ppf(1 - alpha/2, n - 1)
    se = s / np.sqrt(n)
    return m, m - t * se, m + t * se


def var_ci(x, alpha=0.05):
    """Sample variance and CI using chi-square distribution."""
    x = np.asarray(x)
    n = len(x)
    if n < 2:
        return np.nan, np.nan, np.nan
    v = x.var(ddof=1)
    l = (n - 1) * v / chi2.ppf(1 - alpha/2, n - 1)
    u = (n - 1) * v / chi2.ppf(alpha/2, n - 1)
    return v, l, u


def prop_ci(count, n, alpha=0.05):
    """Proportion and Wald CI."""
    if n == 0:
        return np.nan, np.nan, np.nan
    p = count / n
    z = stats.norm.ppf(1 - alpha/2)
    se = np.sqrt(p * (1 - p) / n)
    return p, p - z * se, p + z * se


def kde_moments(x, pdf):
    """Compute mean and variance from discretized pdf."""
    if len(x) == 0:
        return np.nan, np.nan
    pdf = pdf / np.trapz(pdf, x)
    mean = np.trapz(x * pdf, x)
    mean_sq = np.trapz((x ** 2) * pdf, x)
    var = mean_sq - mean ** 2
    return mean, var


def roc_with_best_threshold(x, y):
    """Compute ROC, AUC and threshold maximizing Youden's J (tpr - fpr)."""
    fpr, tpr, thr = roc_curve(y, x)
    J = tpr - fpr
    idx = np.argmax(J)
    return fpr, tpr, thr, thr[idx]


def diag_from_indicator(df, feat, positive_level, indicator_col, positive_value):
    """Diagnostic metrics 1-vs-rest based on a simple categorical rule."""
    d = df[[indicator_col, "Obesity4"]].dropna()
    y_true = (d["Obesity4"] == positive_level).astype(int).values
    y_pred = (d[indicator_col] == positive_value).astype(int).values

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    n = tp + tn + fp + fn

    acc = (tp + tn) / n if n > 0 else np.nan
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    kappa = cohen_kappa_score(y_true, y_pred)

    return {
        "feature": feat,
        "positive_level": positive_level,
        "rule": f"{indicator_col} == {positive_value}",
        "accuracy": acc,
        "sensitivity": sens,
        "specificity": spec,
        "kappa": kappa,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


## 3. Marginal distributions (PMF/PDF and CDF)

In [5]:
# Compute marginal supports and empirical distributions
support = {}
marginals = {}

for col in all_vars:
    s = df[col]
    if col in continuous:
        support[col] = {
            "type": "continuous",
            "min": float(s.min()),
            "max": float(s.max()),
        }
        x, pdf, cdf, _ = kde_pdf_cdf(s)
        marginals[col] = {"x": x, "pdf": pdf, "cdf": cdf}
    else:
        support[col] = {
            "type": "discrete",
            "values": sorted(s.dropna().unique().tolist()),
        }
        pmf, cdf = pmf_cdf(s)
        marginals[col] = {"pmf": pmf, "cdf": cdf}

support_df = pd.DataFrame.from_dict(support, orient="index")
support_df.index.name = "feature"
support_df.reset_index(inplace=True)

support_df.to_csv("tables/support_summary.csv", index=False)
support_df


C:\Users\eldeivid\AppData\Local\Temp\ipykernel_76028\3651750647.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  pdf = pdf / np.trapz(pdf, x)


,feature,type,min,max,values
0,Age,continuous,14.00,61.00,NaN
1,Height,continuous,1.45,1.98,NaN
2,Weight,continuous,39.00,173.00,NaN
3,FCVC,continuous,1.00,3.00,NaN
4,NCP,continuous,1.00,4.00,NaN
5,CH2O,continuous,1.00,3.00,NaN
6,FAF,continuous,0.00,3.00,NaN
7,TUE,continuous,0.00,2.00,NaN
8,FAVC,discrete,NaN,NaN,"[no, yes]"
9,SMOKE,discrete,NaN,NaN,"[no, yes]"


In [6]:
# Plot marginal PDFs/PMFs and CDFs feature-by-feature

for col in all_vars:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    if col in continuous:
        x = marginals[col]["x"]
        pdf = marginals[col]["pdf"]
        cdf = marginals[col]["cdf"]

        # Left: histogram + KDE
        axes[0].hist(df[col].dropna(), bins=30, density=True, alpha=0.5)
        axes[0].plot(x, pdf)
        axes[0].set_title(f"{col} - Marginal PDF")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Density")

        # Right: CDF
        axes[1].plot(x, cdf)
        axes[1].set_title(f"{col} - Marginal CDF")
        axes[1].set_xlabel(col)
        axes[1].set_ylabel("Cumulative probability")

    else:
        pmf = marginals[col]["pmf"]
        cdf = marginals[col]["cdf"]

        # PMF
        axes[0].bar(pmf.index.astype(str), pmf.values)
        axes[0].set_title(f"{col} - Marginal PMF")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Probability")
        axes[0].tick_params(axis='x', rotation=45)

        # CDF
        axes[1].step(cdf.index.astype(str), cdf.values, where="mid")
        axes[1].set_title(f"{col} - Marginal CDF")
        axes[1].set_xlabel(col)
        axes[1].set_ylabel("Cumulative probability")
        axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(f"figures/marginal_{col}.png", dpi=300)
    plt.close(fig)

print("Saved marginal figures to figures/marginal_*.png")


Saved marginal figures to figures/marginal_*.png


## 4. Conditional distributions by obesity level

In [7]:
# Compute conditional PDFs/PMFs and CDFs for each feature given Obesity4
cond = {}

for col in all_vars:
    cond[col] = {}
    for lvl, sub in df.groupby("Obesity4", observed=False):
        s = sub[col].dropna()
        if col in continuous:
            x, pdf, cdf, _ = kde_pdf_cdf(s)
            cond[col][lvl] = {"x": x, "pdf": pdf, "cdf": cdf}
        else:
            pmf, cdf_vals = pmf_cdf(s)
            cond[col][lvl] = {"pmf": pmf, "cdf": cdf_vals}

list(cond.keys())[:5]


C:\Users\eldeivid\AppData\Local\Temp\ipykernel_76028\3651750647.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  pdf = pdf / np.trapz(pdf, x)


['Age', 'Height', 'Weight', 'FCVC', 'NCP']

In [8]:
# Colors for obesity levels
colors = {
    "Insufficient": "tab:blue",
    "Normal": "tab:orange",
    "Overweight": "tab:green",
    "Obesity": "tab:red",
}

# Conditional PDFs/CDFs for continuous variables
for col in continuous:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # PDFs
    for lvl in df["Obesity4"].cat.categories:
        c = cond[col][lvl]
        x, pdf = c["x"], c["pdf"]
        axes[0].plot(x, pdf, label=lvl, color=colors.get(lvl, None))
    axes[0].set_title(f"{col} - Conditional PDFs by Obesity4")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Density")
    axes[0].legend()

    # CDFs
    for lvl in df["Obesity4"].cat.categories:
        c = cond[col][lvl]
        x, cdf_vals = c["x"], c["cdf"]
        axes[1].plot(x, cdf_vals, label=lvl, color=colors.get(lvl, None))
    axes[1].set_title(f"{col} - Conditional CDFs by Obesity4")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Cumulative probability")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"figures/conditional_{col}.png", dpi=300)
    plt.close(fig)

print("Saved conditional continuous figures to figures/conditional_*.png")


Saved conditional continuous figures to figures/conditional_*.png


In [9]:
# Conditional proportions for categorical/binary variables
for col in binary + categorical:
    tab = (
        df.groupby(["Obesity4", col])
        .size()
        .reset_index(name="count")
    )
    total = tab.groupby("Obesity4")["count"].transform("sum")
    tab["prop"] = tab["count"] / total

    fig, ax = plt.subplots(figsize=(8, 4))
    categories_vals = sorted(df[col].dropna().unique().tolist())
    x = np.arange(len(df["Obesity4"].cat.categories))
    width = 0.8 / max(len(categories_vals), 1)

    for j, cat in enumerate(categories_vals):
        sub = tab[tab[col] == cat]
        vals = []
        for lvl in df["Obesity4"].cat.categories:
            row = sub[sub["Obesity4"] == lvl]
            if len(row) == 0:
                vals.append(0.0)
            else:
                vals.append(row["prop"].iloc[0])
        offset = (j - (len(categories_vals)-1)/2) * width
        ax.bar(x + offset, vals, width=width, label=str(cat))

    ax.set_xticks(x)
    ax.set_xticklabels(df["Obesity4"].cat.categories)
    ax.set_ylabel("Proportion")
    ax.set_title(f"{col} - Conditional proportions by Obesity4")
    ax.legend(title=col, bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    plt.savefig(f"figures/conditional_{col}.png", dpi=300)
    plt.close(fig)

print("Saved conditional categorical figures to figures/conditional_*.png")


C:\Users\eldeivid\AppData\Local\Temp\ipykernel_76028\1315307675.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["Obesity4", col])
C:\Users\eldeivid\AppData\Local\Temp\ipykernel_76028\1315307675.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  total = tab.groupby("Obesity4")["count"].transform("sum")
C:\Users\eldeivid\AppData\Local\Temp\ipykernel_76028\1315307675.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this

Saved conditional categorical figures to figures/conditional_*.png


## 5. Continuous features: descriptive statistics and ANOVA

In [10]:
# Descriptive stats (mean, var, CIs) by feature and obesity level
rows = []

for col in continuous:
    for lvl, sub in df.groupby("Obesity4", observed=False):
        x = sub[col].dropna().values
        n = len(x)
        mean, lo_m, hi_m = mean_ci(x)
        var, lo_v, hi_v = var_ci(x)
        rows.append({
            "feature": col,
            "obesity_level": lvl,
            "n": n,
            "mean": mean,
            "mean_ci_low": lo_m,
            "mean_ci_high": hi_m,
            "var": var,
            "var_ci_low": lo_v,
            "var_ci_high": hi_v,
            "min": x.min() if n > 0 else np.nan,
            "max": x.max() if n > 0 else np.nan,
        })

cont_stats = pd.DataFrame(rows)
cont_stats.to_csv("tables/continuous_descriptives.csv", index=False)
cont_stats


,feature,obesity_level,n,mean,mean_ci_low,mean_ci_high,var,var_ci_low,var_ci_high,min,max
0,Age,Insufficient,272,19.783237,19.464569,20.101906,7.126287,6.063621,8.496624,16.00,39.00
1,Age,Normal,287,21.738676,21.146511,22.330841,25.976926,22.194531,30.821435,14.00,61.00
2,Age,Overweight,580,25.207328,24.605992,25.808663,54.368514,48.611529,61.217989,16.00,56.00
3,Age,Obesity,972,25.806181,25.433615,26.178747,35.034324,32.115475,38.372118,15.00,52.00
4,Height,Insufficient,272,1.691117,1.679217,1.703017,0.009937,0.008456,0.011848,1.52,1.90
5,Height,Normal,287,1.676585,1.665603,1.687568,0.008935,0.007634,0.010601,1.50,1.93
6,Height,Overweight,580,1.695792,1.688204,1.703380,0.008656,0.007740,0.009747,1.45,1.93
7,Height,Obesity,972,1.715553,1.709951,1.721155,0.007920,0.007260,0.008675,1.50,1.98
8,Weight,Insufficient,272,49.906330,49.188812,50.623849,36.128607,30.741137,43.075892,39.00,65.00
9,Weight,Normal,287,62.155052,61.074996,63.235109,86.416189,73.833480,102.532188,42.30,87.00


In [ ]:

continuous_vars = ["Age", "Height", "FCVC", "NCP", "CH2O", "FAF", "TUE"]

import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("figures/boxplots", exist_ok=True)

for col in continuous_vars:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x="Obesity4", y=col, order=sorted(df["Obesity4"].unique()))
    plt.title(f"Boxplot of {col} by Obesity Level")
    plt.xlabel("Obesity Level")
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(f"figures/boxplots/boxplot_{col}.png", dpi=300)
    plt.close()


In [11]:
# One-way ANOVA for each continuous feature across Obesity4 levels
anova_rows = []

for col in continuous:
    d = df[[col, "Obesity4"]].dropna()
    groups = [g[col].values for _, g in d.groupby("Obesity4", observed=False)]
    F, p = stats.f_oneway(*groups)
    anova_rows.append({"feature": col, "F": F, "p": p})

anova_df = pd.DataFrame(anova_rows)
anova_df["significant_0.05"] = anova_df["p"] < 0.05
anova_df.to_csv("tables/anova_continuous.csv", index=False)
anova_df


,feature,F,p,significant_0.05
0,Age,94.892632,1.319997e-57,True
1,Height,16.364722,1.632417e-10,True
2,Weight,1992.518260,0.000000e+00,True
3,FCVC,33.269430,5.253959e-21,True
4,NCP,20.369308,5.190784e-13,True
5,CH2O,15.355907,6.958861e-10,True
6,FAF,23.588062,5.152178e-15,True
7,TUE,10.901089,4.186742e-07,True


In [12]:
# Tukey HSD for features with significant ANOVA; apply FDR across all Tukey tests
tuk_list = []

for col in continuous:
    if anova_df.loc[anova_df["feature"] == col, "p"].values[0] < 0.05:
        d = df[[col, "Obesity4"]].dropna()
        t = pairwise_tukeyhsd(endog=d[col], groups=d["Obesity4"], alpha=0.05)
        res = pd.DataFrame(t._results_table.data[1:], columns=t._results_table.data[0])
        res["feature"] = col
        tuk_list.append(res)

if tuk_list:
    tuk_df = pd.concat(tuk_list, ignore_index=True)
    # FDR on Tukey p-values
    rej, p_adj, _, _ = multipletests(tuk_df["p-adj"].values, alpha=0.05, method="fdr_bh")
    tuk_df["p_fdr"] = p_adj
    tuk_df["significant_fdr"] = rej
    tuk_df.to_csv("tables/tukey_continuous_fdr.csv", index=False)
else:
    tuk_df = pd.DataFrame(columns=["group1", "group2", "meandiff", "p-adj", "lower", "upper", "reject", "feature", "p_fdr", "significant_fdr"])

tuk_df


,group1,group2,meandiff,p-adj,lower,upper,reject,feature,p_fdr,significant_fdr
0,Insufficient,Normal,1.9554,0.0006,0.6586,3.2523,True,Age,0.001029,True
1,Insufficient,Obesity,6.0229,0.0000,4.9717,7.0742,True,Age,0.000000,True
2,Insufficient,Overweight,5.4241,0.0000,4.2979,6.5503,True,Age,0.000000,True
3,Normal,Obesity,4.0675,0.0000,3.0380,5.0970,True,Age,0.000000,True
4,Normal,Overweight,3.4687,0.0000,2.3626,4.5747,True,Age,0.000000,True
5,Obesity,Overweight,-0.5989,0.2220,-1.4029,0.2052,False,Age,0.284084,False
6,Insufficient,Normal,-0.0145,0.2456,-0.0346,0.0056,False,Height,0.302277,False
7,Insufficient,Obesity,0.0244,0.0007,0.0082,0.0407,True,Height,0.001159,True
8,Insufficient,Overweight,0.0047,0.9013,-0.0128,0.0221,False,Height,0.983236,False
9,Normal,Obesity,0.0390,0.0000,0.0230,0.0549,True,Height,0.000000,True


## 6. Binary and categorical features: prevalences and chi-square tests

In [13]:
# Conditional prevalences and CIs for each category and obesity level
cat_rows = []

for col in binary + categorical:
    for lvl, sub in df.groupby("Obesity4", observed=False):
        total = sub[col].notna().sum()
        counts = sub[col].value_counts()
        for cat, c in counts.items():
            p, lo, hi = prop_ci(c, total)
            cat_rows.append({
                "feature": col,
                "obesity_level": lvl,
                "category": cat,
                "count": c,
                "total": total,
                "prop": p,
                "prop_ci_low": lo,
                "prop_ci_high": hi,
            })

cat_cond_df = pd.DataFrame(cat_rows)
cat_cond_df.to_csv("tables/categorical_prevalence.csv", index=False)
cat_cond_df


,feature,obesity_level,category,count,total,prop,prop_ci_low,prop_ci_high
0,FAVC,Insufficient,yes,221,272,0.812500,0.766115,0.858885
1,FAVC,Insufficient,no,51,272,0.187500,0.141115,0.233885
2,FAVC,Normal,yes,208,287,0.724739,0.673065,0.776412
3,FAVC,Normal,no,79,287,0.275261,0.223588,0.326935
4,FAVC,Overweight,yes,484,580,0.834483,0.804237,0.864729
...,...,...,...,...,...,...,...,...
82,MTRANS,Obesity,Public_Transportation,759,972,0.780864,0.754859,0.806869
83,MTRANS,Obesity,Automobile,206,972,0.211934,0.186242,0.237626
84,MTRANS,Obesity,Walking,3,972,0.003086,-0.000401,0.006574
85,MTRANS,Obesity,Motorbike,3,972,0.003086,-0.000401,0.006574


In [14]:
# Chi-square tests for association between each categorical/binary feature and Obesity4
chi_rows = []

for col in binary + categorical:
    tab = pd.crosstab(df["Obesity4"], df[col])
    chi2_stat, p, dof, exp = stats.chi2_contingency(tab)
    chi_rows.append({
        "feature": col,
        "chi2": chi2_stat,
        "df": dof,
        "p": p,
        "method": "chi2",
    })

chi_df = pd.DataFrame(chi_rows)
rej, p_adj, _, _ = multipletests(chi_df["p"].values, alpha=0.05, method="fdr_bh")
chi_df["p_fdr"] = p_adj
chi_df["significant_fdr"] = rej

chi_df.to_csv("tables/chi_square_categoricals.csv", index=False)
chi_df


,feature,chi2,df,p,method,p_fdr,significant_fdr
0,FAVC,186.518899,3,3.447840e-40,chi2,9.194239e-40,True
1,SMOKE,13.901526,3,3.042303e-03,chi2,3.042303e-03,True
2,SCC,79.642730,3,3.661575e-17,chi2,5.858521e-17,True
3,Gender,32.196306,3,4.757888e-07,chi2,5.437587e-07,True
4,family_history_with_overweight,575.571169,3,1.990907e-124,chi2,7.963626e-124,True
5,CAEC,715.051862,9,4.014705e-148,chi2,3.211764e-147,True
6,CALC,90.328875,9,1.398545e-15,chi2,1.864726e-15,True
7,MTRANS,150.150517,12,5.284884e-26,chi2,1.056977e-25,True


## 7. Model-based conditional moments vs sample moments

In [15]:
model_rows = []

for col in continuous:
    for lvl in df["Obesity4"].cat.categories:
        c = cond[col][lvl]
        x, pdf = c["x"], c["pdf"]
        m_model, v_model = kde_moments(x, pdf)
        model_rows.append({
            "feature": col,
            "obesity_level": lvl,
            "model_mean": m_model,
            "model_var": v_model,
        })

model_df = pd.DataFrame(model_rows)

# Merge with empirical stats
model_compare = cont_stats.merge(
    model_df,
    on=["feature", "obesity_level"],
    how="left"
)

model_compare["mean_diff"] = model_compare["model_mean"] - model_compare["mean"]
model_compare["var_diff"] = model_compare["model_var"] - model_compare["var"]

model_compare.to_csv("tables/model_vs_sample_moments.csv", index=False)
model_compare


C:\Users\eldeivid\AppData\Local\Temp\ipykernel_76028\3651750647.py:71: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  pdf = pdf / np.trapz(pdf, x)
C:\Users\eldeivid\AppData\Local\Temp\ipykernel_76028\3651750647.py:72: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  mean = np.trapz(x * pdf, x)
C:\Users\eldeivid\AppData\Local\Temp\ipykernel_76028\3651750647.py:73: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  mean_sq = np.trapz((x ** 2) * pdf, x)


,feature,obesity_level,n,mean,mean_ci_low,mean_ci_high,var,var_ci_low,var_ci_high,min,max,model_mean,model_var,mean_diff,var_diff
0,Age,Insufficient,272,19.783237,19.464569,20.101906,7.126287,6.063621,8.496624,16.00,39.00,19.787313,7.668218,0.004076,0.541931
1,Age,Normal,287,21.738676,21.146511,22.330841,25.976926,22.194531,30.821435,14.00,61.00,21.731627,28.079748,-0.007048,2.102823
2,Age,Overweight,580,25.207328,24.605992,25.808663,54.368514,48.611529,61.217989,16.00,56.00,25.250831,56.922482,0.043503,2.553968
3,Age,Obesity,972,25.806181,25.433615,26.178747,35.034324,32.115475,38.372118,15.00,52.00,25.805552,37.118941,-0.000629,2.084617
4,Height,Insufficient,272,1.691117,1.679217,1.703017,0.009937,0.008456,0.011848,1.52,1.90,1.695184,0.009844,0.004067,-0.000094
5,Height,Normal,287,1.676585,1.665603,1.687568,0.008935,0.007634,0.010601,1.50,1.93,1.677835,0.009423,0.001250,0.000488
6,Height,Overweight,580,1.695792,1.688204,1.703380,0.008656,0.007740,0.009747,1.45,1.93,1.695799,0.009228,0.000007,0.000572
7,Height,Obesity,972,1.715553,1.709951,1.721155,0.007920,0.007260,0.008675,1.50,1.98,1.715641,0.008360,0.000088,0.000439
8,Weight,Insufficient,272,49.906330,49.188812,50.623849,36.128607,30.741137,43.075892,39.00,65.00,49.969491,38.289738,0.063161,2.161131
9,Weight,Normal,287,62.155052,61.074996,63.235109,86.416189,73.833480,102.532188,42.30,87.00,62.150294,92.462345,-0.004758,6.046155


## 8. ROC analysis, bootstrap, and discriminative intervals for continuous features

In [16]:
def eval_ovr(df, feat, lvl, n_boot=1000, seed=0):
    """One-vs-rest evaluation with bootstrap AUC, best threshold, errors, KS."""
    rng = np.random.default_rng(seed)
    d = df[[feat, "Obesity4"]].dropna()
    x = d[feat].values
    y = (d["Obesity4"] == lvl).astype(int).values

    # ROC and best threshold
    fpr, tpr, thr, best_thr = roc_with_best_threshold(x, y)
    auc0 = roc_auc_score(y, x)

    # Bootstrap AUC
    aucs = []
    n = len(x)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        xb = x[idx]
        yb = y[idx]
        try:
            aucs.append(roc_auc_score(yb, xb))
        except ValueError:
            continue
    if len(aucs) > 0:
        lo, hi = np.percentile(aucs, [2.5, 97.5])
    else:
        lo, hi = np.nan, np.nan

    # Confusion matrix at best threshold
    y_pred = (x >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    alpha = fp / (fp + tn) if (fp + tn) > 0 else np.nan  # Type I error
    beta = fn / (fn + tp) if (fn + tp) > 0 else np.nan   # Type II error
    power = 1 - beta if not np.isnan(beta) else np.nan

    # KS test between positive and negative class
    pos = x[y == 1]
    neg = x[y == 0]
    ks_stat, ks_p = stats.ks_2samp(pos, neg)

    # Discriminative interval (a, b]
    if pos.mean() >= neg.mean():
        A, B = best_thr, float("inf")
    else:
        A, B = float("-inf"), best_thr

    return {
        "feature": feat,
        "level": lvl,
        "auc": auc0,
        "auc_ci_low": lo,
        "auc_ci_high": hi,
        "threshold": best_thr,
        "a": A,
        "b": B,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "alpha": alpha,
        "beta": beta,
        "power": power,
        "ks": ks_stat,
        "ks_p": ks_p,
        "fpr": fpr,
        "tpr": tpr,
    }


In [17]:
disc_list = []

for feat in continuous:
    for lvl in df["Obesity4"].cat.categories:
        disc_list.append(eval_ovr(df, feat, lvl, n_boot=1000, seed=42))

disc_df = pd.DataFrame(disc_list)
disc_df.to_csv("tables/roc_full_continuous_raw.csv", index=False)
disc_df.head()


,feature,level,auc,auc_ci_low,auc_ci_high,threshold,a,b,tn,fp,fn,tp,alpha,beta,power,ks,ks_p,fpr,tpr
0,Age,Insufficient,0.203682,0.181069,0.227801,16.000000,-inf,16.000000,2,1837,0,272,0.998912,0.000000,1.000000,0.473845,4.736574e-49,"[0.0, 0.000543773790103317, 0.0027188689505165...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,Age,Normal,0.339387,0.307372,0.370370,61.000000,-inf,61.000000,1824,0,286,1,0.000000,0.996516,0.003484,0.291615,3.862624e-19,"[0.0, 0.0, 0.0021929824561403508, 0.0032894736...","[0.0, 0.003484320557491289, 0.0034843205574912..."
2,Age,Overweight,0.527747,0.499436,0.558485,26.047077,26.047077,inf,1225,306,393,187,0.199869,0.677586,0.322414,0.122544,5.743854e-06,"[0.0, 0.0006531678641410843, 0.000653167864141...","[0.0, 0.0, 0.006896551724137931, 0.01034482758..."
3,Age,Obesity,0.687571,0.663476,0.710410,22.200779,22.200779,inf,720,419,280,692,0.367867,0.288066,0.711934,0.344068,1.402693e-55,"[0.0, 0.000877963125548727, 0.0043898156277436...","[0.0, 0.0, 0.0, 0.0, 0.00102880658436214, 0.00..."
4,Height,Insufficient,0.468953,0.433069,0.504423,1.686936,-inf,1.686936,809,1030,108,164,0.560087,0.397059,0.602941,0.144378,8.914262e-05,"[0.0, 0.000543773790103317, 0.0032626427406199...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.003676470588235294..."


In [ ]:

auc_table = disc_df[["feature", "level", "auc", "auc_ci_low", "auc_ci_high"]].copy()
error_table = disc_df[[
    "feature", "level", "tn", "fp", "fn", "tp",
    "alpha", "beta", "power", "threshold", "a", "b"
]].copy()
ks_table = disc_df[["feature", "level", "ks", "ks_p"]].copy()

rej, p_adj, _, _ = multipletests(ks_table["ks_p"].values, alpha=0.05, method="fdr_bh")
ks_table["ks_p_fdr"] = p_adj
ks_table["significant_fdr"] = rej

auc_table.to_csv("tables/roc_auc_continuous.csv", index=False)
error_table.to_csv("tables/roc_errors_continuous.csv", index=False)
ks_table.to_csv("tables/ks_tests_continuous.csv", index=False)

auc_table.head(), error_table.head(), ks_table.head()


(  feature         level       auc  auc_ci_low  auc_ci_high
 0     Age  Insufficient  0.203682    0.181069     0.227801
 1     Age        Normal  0.339387    0.307372     0.370370
 2     Age    Overweight  0.527747    0.499436     0.558485
 3     Age       Obesity  0.687571    0.663476     0.710410
 4  Height  Insufficient  0.468953    0.433069     0.504423,
   feature         level    tn    fp   fn   tp     alpha      beta     power  \
 0     Age  Insufficient     2  1837    0  272  0.998912  0.000000  1.000000   
 1     Age        Normal  1824     0  286    1  0.000000  0.996516  0.003484   
 2     Age    Overweight  1225   306  393  187  0.199869  0.677586  0.322414   
 3     Age       Obesity   720   419  280  692  0.367867  0.288066  0.711934   
 4  Height  Insufficient   809  1030  108  164  0.560087  0.397059  0.602941   
 
    threshold          a          b  
 0  16.000000       -inf  16.000000  
 1  61.000000       -inf  61.000000  
 2  26.047077  26.047077        inf  
 3  2

In [19]:
# ROC curves per continuous feature (one figure per feature with 4 curves)

for feat in continuous:
    fig, ax = plt.subplots(figsize=(6, 5))
    for lvl in df["Obesity4"].cat.categories:
        row = disc_df[(disc_df["feature"] == feat) & (disc_df["level"] == lvl)].iloc[0]
        fpr = row["fpr"]
        tpr = row["tpr"]
        auc_val = row["auc"]
        ax.plot(fpr, tpr, label=f"{lvl} (AUC={auc_val:.2f})")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curves for {feat} (one-vs-rest)")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"figures/roc_{feat}.png", dpi=300)
    plt.close(fig)

print("Saved ROC figures to figures/roc_*.png")


Saved ROC figures to figures/roc_*.png


## 9. Diagnostic metrics for selected categorical features

In [ ]:
cat_diag_list = []

# 1) FAVC == 'yes' to detect Obesity
cat_diag_list.append(
    diag_from_indicator(df, "FAVC", "Obesity", "FAVC", "yes")
)

# 2) family_history_with_overweight == 'yes' to detect Obesity
cat_diag_list.append(
    diag_from_indicator(df, "family_history_with_overweight", "Obesity",
                        "family_history_with_overweight", "yes")
)

# 3) CAEC == 'Frequently' to detect Obesity
cat_diag_list.append(
    diag_from_indicator(df, "CAEC", "Obesity", "CAEC", "Frequently")
)


cat_diag_df = pd.DataFrame(cat_diag_list)
cat_diag_df.to_csv("tables/categorical_diagnostics.csv", index=False)
cat_diag_df


,feature,positive_level,rule,accuracy,sensitivity,specificity,kappa,tn,fp,fn,tp
0,FAVC,Obesity,FAVC == yes,0.558503,0.980453,0.198420,0.167573,226,913,19,953
1,family_history_with_overweight,Obesity,family_history_with_overweight == yes,0.635244,0.991770,0.330992,0.305394,377,762,8,964
2,CAEC,Obesity,CAEC == Frequently,0.432496,0.008230,0.794557,-0.208704,905,234,964,8
